In [1]:
# For reading data
import os
import numpy as np
from xml.dom import minidom

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# For visualizing
import plotly.express as px
from torchvision.io import read_image
from torchvision import tv_tensors
from torchvision.transforms.v2 import functional as F

# For model building
import torch
import torch.nn as nn

# for videos
import cv2 as cv

In [2]:
### THIS IS THE PROFESSOR'S EXACT CODE, IT IS NOT TAILORED TO OUR DATA YET. THIS IS TO BE USED AS A STARTING POINT ###

class BaseballVideos(torch.utils.data.Dataset):
    def __init__(self, root=None, transforms=None):
        self.root = root if root is not None else os.getcwd()
        self.transforms = transforms

        video_dir = os.path.join(self.root, "Raw Videos")
        annot_dir = os.path.join(self.root, "Annotations")

        # get sorted file lists
        self.vids = sorted([f for f in os.listdir(video_dir) if f.endswith(".mov")])
        self.notes_files = sorted([f for f in os.listdir(annot_dir) if f.endswith(".xml")])

        if len(self.vids) != len(self.notes_files):
            raise RuntimeError("Mismatch of annotation files and video files.")

        imgs = []
        notes = []

        for vid_file, xml_file in zip(self.vids, self.notes_files):
            vid_path = os.path.join(video_dir, vid_file)
            xml_path = os.path.join(annot_dir, xml_file)

            cap = cv.VideoCapture(vid_path)
            note = minidom.parse(xml_path)

            ret = True
            frame_count = 0

            # read frames
            while ret:
                ret, frame = cap.read()
                if ret:
                    frame_count += 1
                    frame = np.moveaxis(frame, -1, 0)
                    imgs.append(torch.from_numpy(frame))
                    canvas_size = list(frame.shape[1:])

            # process annotations per frame
            for f in range(frame_count):
                frame_i = [
                    j for j in note.getElementsByTagName("box")
                    if int(j.attributes['frame'].value) == f
                ]

                boxes, labels, areas, movings = [], [], [], []

                for j in frame_i:
                    moving = j.getElementsByTagName('attribute')[0].firstChild.data == 'true'

                    xtl = float(j.attributes['xtl'].value)
                    ytl = float(j.attributes['ytl'].value)
                    xbr = float(j.attributes['xbr'].value)
                    ybr = float(j.attributes['ybr'].value)

                    box = (xtl, ytl, xbr, ybr)
                    area = (xbr - xtl) * (ybr - ytl)

                    boxes.append(box)
                    labels.append('baseball')
                    areas.append(area)
                    movings.append(moving)

                target = {
                    "boxes": tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=canvas_size),
                    "labels": labels,
                    "area": areas,
                    "moving": movings
                }

                notes.append(target)

        self.imgs = imgs
        self.notes = notes

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = self.imgs[idx]
        target = self.notes[idx]

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target